# Black-Box Adversarial Generator

This examples show how to use ZOO attack in order to generate adversarial examples. The results are evaluated using evaluation metrics such as F1 score, Accuracy, TPR, FPR. 

### Import Libraries


In [11]:
import numpy as np 
import pandas as pd 
import json
#import xgboost as xgb
from art.estimators.classification import SklearnClassifier
from sklearn.ensemble import RandomForestClassifier
from art.attacks.evasion import ZooAttack
from sklearn.preprocessing import LabelEncoder, StandardScaler
from utils import load_config
import matplotlib.pyplot as plt
import seaborn as sns

import warnings
warnings.filterwarnings('ignore')

In [12]:
def preprocess(df_test, columns_to_drop):
    df_test_reduced = df_test.drop(columns_to_drop, axis=1)
    df_test_reduced.replace([np.inf, -np.inf], np.nan, inplace=True)
    df_test_no_na = df_test_reduced.dropna(axis=0)

    y_test = df_test_no_na['Label']
    X_test = df_test_no_na.drop(['Label'], axis=1)

    le_y = LabelEncoder()
    y_test = le_y.fit_transform(y_test.values)
    y_test = pd.DataFrame(y_test)

    scaler = StandardScaler()
    X_scaled_test = scaler.fit_transform(X_test)
    X_scaled_test = pd.DataFrame(X_scaled_test, columns=X_test.columns)

    return X_scaled_test, y_test, scaler

In [13]:
config_path = 'config.json'

# Load the configuration
config = load_config(config_path)

### Read the Configuration file

In [14]:
# Access configuration parameters
dataset_path = config.get("dataset_path")
eps = config.get("eps")
output_path = config.get("output_path")
adversarial_dataset_output_path = config.get("adversarial_dataset_output_path")
columns_to_drop = config.get("columns_to_drop")
type_of_attack = config.get("type_of_attack")
jsma_theta = config.get("jsma_theta", 1.0)
jsma_gamma = config.get("jsma_gamma", 0.1)

### Print the inforamtion accessed from the Config file

In [15]:
# Print the configuration parameters to verify
print(f"Dataset path: {dataset_path}")
print(f"Epsilon (eps): {eps}")
print(f"Output path: {output_path}")
print(f"Adversarial dataset output path: {adversarial_dataset_output_path}")
print(f"Columns to drop: {columns_to_drop}")
print(f"Type of attack chosen: {type_of_attack}")
print(f"JSMA theta: {jsma_theta}")
print(f"JSMA gamma: {jsma_gamma}")

Dataset path: dataset/test.csv
Epsilon (eps): 0.5
Output path: None
Adversarial dataset output path: results/adversarial_dataset.csv
Columns to drop: ['Flow ID', 'Src IP', 'Src Port', 'Dst IP', 'Dst Port', 'Protocol', 'Timestamp']
Type of attack chosen: ['zoo']
JSMA theta: 1.0
JSMA gamma: 0.1


### Load Dataset



In [16]:
# Load dataset
df_test = pd.read_csv(dataset_path)


### Preprocess the data 

In [17]:
# Preprocess the data
X_scaled_test, y_test, scaler = preprocess(df_test, columns_to_drop)
print("Data preprocessed successfully.")

Data preprocessed successfully.


### Function for evaluating the adversarial examples 

Function to define the evaluation metrics in order to evaluate the adversarial examples. The results will be saved in a json file.

In [18]:
from sklearn.metrics import accuracy_score, f1_score, roc_auc_score, confusion_matrix

def evaluation_metrics(y_true, classes, predicted_test, dataset_name):
    metrics = {}

    # Confusion Matrix
    cnf_matrix = confusion_matrix(y_true, classes)
    metrics['confusion_matrix'] = cnf_matrix.tolist()  # convert to list for JSON serialization
    print("Confusion Matrix:\n", cnf_matrix)

    # Plot and save confusion matrix
    plt.figure(figsize=(10, 7))
    sns.heatmap(cnf_matrix, annot=True, fmt='d', cmap='Blues')
    plt.title(f'Confusion Matrix - {dataset_name}')
    plt.xlabel('Predicted')
    plt.ylabel('Actual')
    plt.savefig(f'results/Confusion_Matrix_{dataset_name}.png')
    plt.close()

    FP = cnf_matrix.sum(axis=0) - np.diag(cnf_matrix)
    FN = cnf_matrix.sum(axis=1) - np.diag(cnf_matrix)
    TP = np.diag(cnf_matrix)
    TN = cnf_matrix.sum() - (FP + FN + TP)

    FP = FP.astype(float)
    FN = FN.astype(float)
    TP = TP.astype(float)
    TN = TN.astype(float)

    # Accuracy
    accuracy = accuracy_score(y_true, classes)
    metrics['accuracy'] = accuracy
    print('Accuracy: %f' % accuracy)

    # True positive rate - TPR
    TPR = TP / (TP + FN)
    metrics['TPR'] = np.mean(TPR)
    print("TPR: ", np.mean(TPR))

    # False positive rate - FPR
    FPR = FP / (FP + TN)
    metrics['FPR'] = np.mean(FPR)
    print("FPR: ", np.mean(FPR))

    # F1 Score
    f1 = f1_score(y_true, classes, average='weighted')
    metrics['f1_score'] = f1
    print('F1 score: %f' % f1)

    # AUC Score
    #auc = roc_auc_score(y_true, predicted_test, multi_class='ovr')
    #metrics['auc_score'] = auc
    #print("AUC Score: %f" % auc)

    # Save metrics to JSON
    with open(f'results/Evaluation_Metrics_{dataset_name}.json', 'w') as f:
        json.dump(metrics, f, indent=4)

    return

In [19]:
X_scaled_test = X_scaled_test.to_numpy()

### Define the State of the Art classification model 

In [20]:
model = RandomForestClassifier()
model.fit(X=X_scaled_test, y=y_test)

RandomForestClassifier()

### Wrapping the model to ART classifier in order to be used

In [21]:
# Create ART classifier for scikit-learn RandomForestClassifier
clf_rf = SklearnClassifier(model=model)

### Define ZOO attack 


In [22]:
# Initialize the ZOO Attack with increased impact parameters
attack = ZooAttack(
    classifier=clf_rf,
    confidence=0.5,  # Increase confidence to make the attack stronger
    targeted=False,
    learning_rate=0.01,  # Adjust learning rate as needed
    max_iter=50,  # Increase maximum iterations for more potent attacks
    binary_search_steps=10,  # Adjust binary search steps if needed
    initial_const=1.0,  # Increase initial constant
    abort_early=False,  # Disable early stopping
    use_resize=False,
    use_importance=False,
    nb_parallel=50,
    batch_size=1,
    variable_h=0.1
)
# Generate adversarial examples
X_test_adv = attack.generate(X_scaled_test)

ZOO: 100%|██████████| 364/364 [08:32<00:00,  1.41s/it]


### Save the adversarial Dataset

In [25]:
import pandas as pd

# Assuming scaler, df_test, columns_to_drop, X_scaled_test, y_test are already defined
# and X_test_adv is the generated adversarial examples array

# Step 1: Denormalize the adversarial examples
X_test_adv_denorm = scaler.inverse_transform(X_test_adv)

# Verify the shape of the denormalized array
print("Shape of denormalized adversarial examples:", X_test_adv_denorm.shape)

# Step 2: Convert to DataFrame
# Use the column names from the original DataFrame, excluding the 'Label' column
column_names = df_test.drop(columns=columns_to_drop + ['Label']).columns
print("Expected column names:", column_names)
print("Number of expected columns:", len(column_names))
print("Number of actual columns in X_test_adv_denorm:", X_test_adv_denorm.shape[1])

# Ensure that the length of column_names matches the number of columns in X_test_adv_denorm
if len(column_names) == X_test_adv_denorm.shape[1]:
    X_test_adv_denorm_df = pd.DataFrame(X_test_adv_denorm, columns=column_names)
else:
    raise ValueError("Mismatch between the number of columns in X_test_adv_denorm and column_names")

# Step 3: Reattach dropped columns
dropped_columns = df_test[columns_to_drop].iloc[X_test_adv_denorm_df.index].reset_index(drop=True)
df_adv_complete = pd.concat([dropped_columns, X_test_adv_denorm_df], axis=1)

# Step 4: Add Labels
df_adv_complete['Label'] = y_test.values

# Step 5: Save the complete adversarial dataset
denormalized_adversarial_dataset_path = 'results/denormalized_adversarial_dataset.csv'
df_adv_complete.to_csv(denormalized_adversarial_dataset_path, index=False)
print(f"Denormalized adversarial dataset saved to {denormalized_adversarial_dataset_path}")


Shape of denormalized adversarial examples: (364, 76)
Expected column names: Index(['Flow Duration', 'Total Fwd Packet', 'Total Bwd packets',
       'Total Length of Fwd Packet', 'Total Length of Bwd Packet',
       'Fwd Packet Length Max', 'Fwd Packet Length Min',
       'Fwd Packet Length Mean', 'Fwd Packet Length Std',
       'Bwd Packet Length Max', 'Bwd Packet Length Min',
       'Bwd Packet Length Mean', 'Bwd Packet Length Std', 'Flow Bytes/s',
       'Flow Packets/s', 'Flow IAT Mean', 'Flow IAT Std', 'Flow IAT Max',
       'Flow IAT Min', 'Fwd IAT Total', 'Fwd IAT Mean', 'Fwd IAT Std',
       'Fwd IAT Max', 'Fwd IAT Min', 'Bwd IAT Total', 'Bwd IAT Mean',
       'Bwd IAT Std', 'Bwd IAT Max', 'Bwd IAT Min', 'Fwd PSH Flags',
       'Bwd PSH Flags', 'Fwd URG Flags', 'Bwd URG Flags', 'Fwd Header Length',
       'Bwd Header Length', 'Fwd Packets/s', 'Bwd Packets/s',
       'Packet Length Min', 'Packet Length Max', 'Packet Length Mean',
       'Packet Length Std', 'Packet Length Varian

### Evaluate the adversarial dataset 

In [26]:
# Evaluate on Adversarial Test Data
y_pred_adv = model.predict(X_test_adv)
evaluation_metrics(y_test.values.ravel(), y_pred_adv, y_pred_adv, "Adversarial_Data")


Confusion Matrix:
 [[11  0  0  1 61]
 [ 0 73  0  0  0]
 [ 0  0 73  0  0]
 [ 8  0  0 65  0]
 [ 1 25  2  0 44]]
Accuracy: 0.730769
TPR:  0.7304414003044141
FPR:  0.06721037518241303
F1 score: 0.702417
